In [ ]:
import concurrent.futures
import os
import subprocess

import pandas as pd
from dotenv import load_dotenv
from loguru import logger

load_dotenv()

In [ ]:
PASSWORD_STORE_DIR = os.path.expanduser("~/.password-store")

In [ ]:
def list_gpg_files(folder_path):
    gpg_files = []

    # Walk through the directory tree
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".gpg"):
                # Append the full path of the .gpg file to the list
                gpg_files.append(os.path.join(root, file))

    return gpg_files

In [ ]:
# Function to extract details for each password entry
def extract_password_details(entry_name):
    # Assuming entry_name is the path to the password file
    passphrase = os.getenv("GPG_PASSPHRASE")
    if passphrase is None:
        raise ValueError("GPG_PASSPHRASE environment variable is not set")
    result = subprocess.run(
        [
            "gpg",
            "--batch",
            "--yes",
            "--pinentry-mode",
            "loopback",
            "--passphrase",
            passphrase,
            "--decrypt",
            entry_name,
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        logger.error(f"Failed to decrypt password for {entry_name}")
        logger.error(result.stderr)
        return None

    password_details = result.stdout.splitlines()

    # Here you need to adapt depending on how your passwords are stored/formatted
    details = {
        "name": entry_name,
        "url": "",  # If this info is stored in your entries, extract it
        "email": "",  # If this info is stored in your entries, extract it
        "username": "",  # If this info is stored in your entries, extract it
        "password": password_details[0],  # Assuming the first line is the password
        "note": "\n".join(
            password_details[1:]
        ),  # Remaining lines can be stored as notes
        "totp": "",  # If you use TOTP, extract this if available
        "vault": "Personal",  # You can categorize your vault as needed
    }
    return details

In [ ]:
# Define the template DataFrame
df_template = pd.DataFrame(
    columns=["name", "url", "email", "username", "password", "note", "totp", "vault"]
)
password_entries = list_gpg_files(PASSWORD_STORE_DIR)
logger.info(f"Found {len(password_entries)} password entries.")

if not password_entries:
    raise Exception(
        "Failed to retrieve passwords. Ensure you have GnuPG set up correctly."
    )


In [ ]:
# Parallel processing using ThreadPoolExecutor
with concurrent.futures.ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(extract_password_details, password_entries))

# Filter out None results (in case extract_password_details returns None)
filtered_results = [result for result in results if result]

# Convert the data into a DataFrame
df = pd.DataFrame(filtered_results)

# Ensure it follows the template format
df = pd.concat([df_template, df], ignore_index=True)
df.head()

In [ ]:
# Write the DataFrame to a CSV
df.to_csv("../data/passwords_export.csv", index=False, quoting=1)